# VEP Analysis Only — Python Conversion of MATLAB Workflow

This notebook converts the **VEP-only** branch of `pERG_VEP_analysis_NEW.m` into Python.

It intentionally leaves out all pERG analysis.

## What this notebook does

1. Loads all `.txt` trace files from a selected folder.
2. Reads the 4th column from each file as the VEP waveform, matching the MATLAB line `Twave = table2array(T(:,4))`.
3. Baseline-normalizes each trace by subtracting the first value.
4. Detects VEP components:
   - **N1** = maximum positive peak
   - **P1** = minimum before N1
   - **P2** = minimum after N1
5. Calculates:
   - `Amp_1 = N1 - P1`
   - `Amp_2 = N1 - P2`
   - `Implicit_time_1 = N1_index - P1_index`
   - `Implicit_time_2 = P2_index - N1_index`
6. Saves summary CSV files and figures.

> Note: The original MATLAB script uses dialog boxes to manually reselect N1/P1/P2 windows. This notebook provides optional detection windows instead of pop-up dialogs, which is usually better for reproducible analysis.


## 1. Install/import libraries

Required packages:

```bash
pip install pandas numpy matplotlib ipywidgets openpyxl
```

`ipywidgets` is optional, but useful if you later want interactive controls.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (14, 4)
plt.rcParams["axes.grid"] = True


## 2. Set input/output folders

Put your VEP `.txt` files into one folder and update `input_dir` below.

The MATLAB script used `uigetdir` and then loaded all `*.txt` files from that folder.


In [ ]:
# CHANGE THIS PATH to the folder containing your VEP .txt files
input_dir = Path("./VEP_txt_files")

# Output folder for CSVs and figures
output_dir = Path("./vep_analysis_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

txt_files = sorted(input_dir.glob("*.txt"))
print(f"Found {len(txt_files)} .txt files")
for f in txt_files[:10]:
    print(f.name)


## 3. Helper functions

These functions reproduce the VEP-specific MATLAB logic.

### MATLAB equivalent

```matlab
T = readtable(baseFileName);
Twave = table2array(T(:,4));
Twave_start = Twave(1:1);

if Twave_start < 0
    Twave_norm = Twave + abs(Twave_start);
elseif Twave_start > 0
    Twave_norm = Twave - abs(Twave_start);
end
```

In Python this is simply:

```python
Twave_norm = Twave - Twave[0]
```

because subtracting the first value handles both positive and negative offsets.


In [ ]:
def read_waveform_txt(path, waveform_col=3, expected_len=494):
    """Read one VEP trace from a .txt file.

    Parameters
    ----------
    path : pathlib.Path
        Path to a .txt file.
    waveform_col : int
        Zero-based column index. MATLAB column 4 = Python column index 3.
    expected_len : int or None
        If 494, traces are trimmed/padded behavior is checked. This notebook trims to expected_len.

    Returns
    -------
    raw : np.ndarray
        Raw waveform from column 4.
    norm : np.ndarray
        Baseline-normalized waveform, first value shifted to zero.
    """
    # sep=None lets pandas infer tabs, spaces, or commas using the Python parser.
    df = pd.read_csv(path, sep=None, engine="python", header=None)

    if df.shape[1] <= waveform_col:
        raise ValueError(f"{path.name} has only {df.shape[1]} columns; expected at least {waveform_col + 1}")

    raw = pd.to_numeric(df.iloc[:, waveform_col], errors="coerce").dropna().to_numpy(dtype=float)

    if expected_len is not None:
        if len(raw) < expected_len:
            raise ValueError(f"{path.name} has {len(raw)} valid samples; expected at least {expected_len}")
        raw = raw[:expected_len]

    norm = raw - raw[0]
    return raw, norm


def detect_vep_peaks(trace, windows=None):
    """Detect VEP N1/P1/P2 peaks using the MATLAB logic or optional windows.

    Default MATLAB-style logic:
    - N1 = max(trace)
    - P1 = min(trace before N1)
    - P2 = min(trace after N1)

    Optional windows use Python slice ranges in sample indices:
    windows = {
        "N1": (start, end),
        "P1": (start, end),
        "P2": (start, end),
    }

    Indices returned are zero-based sample indices.
    """
    trace = np.asarray(trace, dtype=float)
    n = len(trace)

    if windows is None:
        n1_idx = int(np.argmax(trace))
        n1_val = float(trace[n1_idx])

        p1_region = trace[:n1_idx + 1]
        p1_idx = int(np.argmin(p1_region))
        p1_val = float(trace[p1_idx])

        p2_region = trace[n1_idx:]
        p2_idx = int(np.argmin(p2_region) + n1_idx)
        p2_val = float(trace[p2_idx])
    else:
        def peak_in_window(name, fn):
            start, end = windows[name]
            start = max(0, int(start))
            end = min(n, int(end))
            if end <= start:
                raise ValueError(f"Invalid {name} window: {windows[name]}")
            region = trace[start:end]
            local_idx = int(fn(region))
            idx = start + local_idx
            return idx, float(trace[idx])

        n1_idx, n1_val = peak_in_window("N1", np.argmax)
        p1_idx, p1_val = peak_in_window("P1", np.argmin)
        p2_idx, p2_val = peak_in_window("P2", np.argmin)

    measurements = {
        "N1_idx": n1_idx,
        "N1_uV": n1_val,
        "P1_idx": p1_idx,
        "P1_uV": p1_val,
        "P2_idx": p2_idx,
        "P2_uV": p2_val,
        "Amp_1_N1_minus_P1": n1_val - p1_val,
        "Amp_2_N1_minus_P2": n1_val - p2_val,
        "Implicit_time_1_N1_minus_P1": n1_idx - p1_idx,
        "Implicit_time_2_P2_minus_N1": p2_idx - n1_idx,
    }
    return measurements


def plot_trace_with_peaks(trace, peaks, title="VEP trace", save_path=None):
    x = np.arange(len(trace))
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(x, trace, linewidth=1)
    ax.plot(peaks["N1_idx"], peaks["N1_uV"], "r*", markersize=12, label="N1 max")
    ax.plot(peaks["P1_idx"], peaks["P1_uV"], "go", markersize=8, label="P1 min before N1")
    ax.plot(peaks["P2_idx"], peaks["P2_uV"], "bx", markersize=8, label="P2 min after N1")
    ax.set_title(title)
    ax.set_xlabel("Sample index / ms if sampling is 1 kHz")
    ax.set_ylabel("uV")
    ax.legend()
    ax.grid(True)
    if save_path is not None:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    return fig, ax


## 4. Optional peak-detection windows

Leave `windows = None` to use the original automatic MATLAB-style peak detection.

To manually constrain detection regions, edit the values below. These are zero-based Python indices. For example, `(50, 150)` means search samples 50 through 149.

This replaces the MATLAB pop-up dialog workflow in a reproducible way.


In [ ]:
# Option A: automatic detection, closest to the default MATLAB branch
windows = None

# Option B: uncomment/edit to constrain peak finding windows
# windows = {
#     "P1": (0, 120),
#     "N1": (80, 250),
#     "P2": (180, 494),
# }


## 5. Run VEP analysis on all traces

This section loads each `.txt` file, extracts column 4, baseline-normalizes the waveform, detects N1/P1/P2, calculates amplitude and implicit-time measurements, and saves individual trace plots.


In [ ]:
all_raw = []
all_norm = []
records = []

individual_plot_dir = output_dir / "individual_traces"
individual_plot_dir.mkdir(exist_ok=True)

for path in txt_files:
    print(f"Reading {path.name}")
    raw, norm = read_waveform_txt(path, waveform_col=3, expected_len=494)
    peaks = detect_vep_peaks(norm, windows=windows)

    all_raw.append(raw)
    all_norm.append(norm)

    record = {"file": path.name}
    record.update(peaks)
    records.append(record)

    plot_trace_with_peaks(
        norm,
        peaks,
        title=f"VEP trace: {path.name}",
        save_path=individual_plot_dir / f"{path.stem}_vep_peaks.png"
    )

measurements_df = pd.DataFrame(records)
measurements_df.head()


## 6. Save measurements

This reproduces the MATLAB `measurements_all` output, but with labeled columns and filenames.


In [ ]:
measurements_csv = output_dir / "vep_measurements.csv"
measurements_df.to_csv(measurements_csv, index=False)
print(f"Saved: {measurements_csv}")
measurements_df


## 7. Average trace and SEM

The MATLAB script calculates:

```matlab
Twave_norm_AVG = mean(Twave_norm_all, 2);
SEM = std(trace_values) / sqrt(number_of_traces)
```

This notebook does the same using NumPy.


In [ ]:
if len(all_norm) == 0:
    raise RuntimeError("No traces loaded. Check input_dir and .txt files.")

Twave_norm_all = np.column_stack(all_norm)  # shape: samples x files
Tfold = np.arange(1, Twave_norm_all.shape[0] + 1)

Twave_norm_AVG = Twave_norm_all.mean(axis=1)
Twave_norm_SEM = Twave_norm_all.std(axis=1, ddof=1) / np.sqrt(Twave_norm_all.shape[1])

Twave_norm_AVG_SEM = pd.DataFrame({
    "sample": Tfold,
    "mean_minus_sem": Twave_norm_AVG - Twave_norm_SEM,
    "mean": Twave_norm_AVG,
    "mean_plus_sem": Twave_norm_AVG + Twave_norm_SEM,
})

avg_sem_csv = output_dir / "vep_average_sem.csv"
Twave_norm_AVG_SEM.to_csv(avg_sem_csv, index=False)
print(f"Saved: {avg_sem_csv}")
Twave_norm_AVG_SEM.head()


## 8. Detect N1/P1/P2 on average trace

This reproduces the average-trace measurement part of the MATLAB VEP branch.


In [ ]:
avg_peaks = detect_vep_peaks(Twave_norm_AVG, windows=windows)
avg_peaks_df = pd.DataFrame([avg_peaks])

avg_peaks_csv = output_dir / "vep_average_trace_measurements.csv"
avg_peaks_df.to_csv(avg_peaks_csv, index=False)
print(f"Saved: {avg_peaks_csv}")
avg_peaks_df


## 9. Plot average trace with SEM and peak labels

Equivalent to the MATLAB `figure101` average trace plot.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.fill_between(
    Tfold,
    Twave_norm_AVG - Twave_norm_SEM,
    Twave_norm_AVG + Twave_norm_SEM,
    alpha=0.3,
    label="SEM"
)
ax.plot(Tfold, Twave_norm_AVG, linewidth=2, label="Mean VEP")

for label, color, marker in [("N1", "red", "*"), ("P1", "green", "o"), ("P2", "blue", "x")]:
    idx = avg_peaks[f"{label}_idx"]
    val = avg_peaks[f"{label}_uV"]
    ax.plot(Tfold[idx], val, marker=marker, color=color, markersize=12, label=label)
    ax.text(Tfold[idx], val, label, color=color, fontsize=12, fontweight="bold")

ax.set_title("Average VEP trace ± SEM")
ax.set_xlabel("Sample index / ms if sampling is 1 kHz")
ax.set_ylabel("uV")
ax.legend()
ax.grid(True)

avg_plot_path = output_dir / "vep_average_trace_sem.png"
fig.savefig(avg_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {avg_plot_path}")


## 10. Plot all traces plus average trace

Equivalent to the MATLAB `figure102` plot.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for i in range(Twave_norm_all.shape[1]):
    ax.plot(Tfold, Twave_norm_all[:, i], linewidth=0.5, alpha=0.5)

ax.plot(Tfold, Twave_norm_AVG, linewidth=3, label="Average trace")
ax.set_title("All VEP traces plus average trace")
ax.set_xlabel("Sample index / ms if sampling is 1 kHz")
ax.set_ylabel("uV")
ax.legend()
ax.grid(True)

all_plot_path = output_dir / "vep_all_traces_plus_average.png"
fig.savefig(all_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {all_plot_path}")


## 11. Export normalized traces matrix

This saves the normalized trace matrix in a Python-friendly CSV format.

Rows = time/sample points.  
Columns = files/traces.


In [ ]:
trace_matrix_df = pd.DataFrame(Twave_norm_all, columns=[p.name for p in txt_files])
trace_matrix_df.insert(0, "sample", Tfold)

trace_matrix_csv = output_dir / "vep_normalized_traces_matrix.csv"
trace_matrix_df.to_csv(trace_matrix_csv, index=False)
print(f"Saved: {trace_matrix_csv}")
trace_matrix_df.head()


## 12. Summary of output files

After running the notebook, check the `vep_analysis_outputs/` folder for:

- `vep_measurements.csv`
- `vep_average_sem.csv`
- `vep_average_trace_measurements.csv`
- `vep_normalized_traces_matrix.csv`
- `vep_average_trace_sem.png`
- `vep_all_traces_plus_average.png`
- `individual_traces/*.png`
